# AI Agent with Lakebase Postgres as Session Memory

Build a simple AI agent that uses a Lakebase Postgres database as persistent session memory, so the agent remembers users across restarts and cold starts. Runs on serverless compute, Python only.

## Why do agents need persistent memory?

Most demo agents keep conversation history in a Python variable:

```python
history.append({"role": "user", "content": "Hi, I'm Alan"})
```

That works — until the process ends. Then:

* **Serverless agents are stateless.** A cold start or a new process has zero memory: the agent forgets the user completely.
* **Agents can't scale horizontally.** Two replicas of your agent each carry their own, divergent copy of the conversation.
* **There's no audit trail.** When the process dies, the history is gone.

The fix: store every turn in a **database**. Lakebase is Databricks' serverless Postgres-compatible database, which gives you:

* **Standard Postgres** — plain SQL and drivers, no new APIs to learn
* **Autoscaling + scale-to-zero** — an idle agent's memory store costs ~nothing
* **Branching** — copy-on-write branches let you test schema changes against real data safely
* **OAuth credentials minted from your Databricks identity** — no passwords in your code

In this demo we point a tiny agent at a Lakebase database, save every conversation turn to a Postgres table, then "restart" the agent and watch it still remember the user.

In [0]:
%pip install pg8000
# NOTE: pg8000 is pure Python. Do NOT install psycopg/psycopg2 here —
# they crash the serverless kernel (SIGABRT).

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import ssl
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  # pre-authenticated with your Databricks identity

LAKEBASE_HOST = "ep-cold-forest-d8ignwgw.database.us-east-2.cloud.databricks.com"
LAKEBASE_DATABASE = "databricks_postgres"
LAKEBASE_ENDPOINT = "projects/alan/branches/production/endpoints/primary"


def get_connection():
    """Open a fresh connection to the Lakebase database.

    Authentication is OAuth: we mint a short-lived token tied to our
    Databricks identity (no passwords anywhere). Tokens expire after
    1 hour, so we open a fresh connection per operation — each call is
    self-contained, which keeps the demo simple and robust.
    """
    import pg8000.native

    token = w.postgres.generate_database_credential(
        endpoint=LAKEBASE_ENDPOINT
    ).token  # NEVER print or log this token

    # The equivalent of sslmode=require: encrypt the connection.
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    return pg8000.native.Connection(
        user=w.current_user.me().user_name,
        host=LAKEBASE_HOST,
        database=LAKEBASE_DATABASE,
        password=token,
        ssl_context=ctx,
        timeout=30,
    )


# Smoke test: one query, then close.
conn = get_connection()
rows = conn.run("SELECT current_database(), current_user, version()")
conn.close()
print("Connected to:", rows[0][0], "| logged in as:", rows[0][1])
print("Server:", rows[0][2])

Connected to: databricks_postgres | logged in as: alan.dennis@databricks.com
Server: PostgreSQL 18.6 (fd34222) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [0]:
# Create the memory schema. DROP first so the demo is re-runnable.
conn = get_connection()
conn.run("DROP TABLE IF EXISTS agent_session_memory")
conn.run(
    """
    CREATE TABLE agent_session_memory (
        session_id  TEXT        NOT NULL,
        turn        INTEGER     NOT NULL,
        role        TEXT        NOT NULL,   -- 'user' or 'agent'
        content     TEXT        NOT NULL,
        created_at  TIMESTAMPTZ DEFAULT now(),
        PRIMARY KEY (session_id, turn)
    )
    """
)
conn.close()
print("Table agent_session_memory created.")

Table agent_session_memory created.


In [0]:
class SessionMemory:
    """Persistent conversation memory backed by a Lakebase Postgres table.

    Every method opens a fresh connection (minting a fresh OAuth token),
    so it works no matter which process — or which replica — calls it.
    """

    def __init__(self, table="agent_session_memory"):
        self.table = table

    def save_turn(self, session_id, turn, role, content):
        """INSERT one turn of the conversation."""
        conn = get_connection()
        conn.run(
            f"INSERT INTO {self.table} (session_id, turn, role, content) "
            "VALUES (:session_id, :turn, :role, :content)",
            session_id=session_id,
            turn=turn,
            role=role,
            content=content,
        )
        conn.close()

    def get_history(self, session_id):
        """SELECT all turns for a session, ordered by turn."""
        conn = get_connection()
        rows = conn.run(
            f"SELECT turn, role, content FROM {self.table} "
            "WHERE session_id = :session_id ORDER BY turn",
            session_id=session_id,
        )
        conn.close()
        return [{"turn": t, "role": r, "content": c} for (t, r, c) in rows]

    def forget_session(self, session_id):
        """DELETE every turn for a session (a full memory wipe)."""
        conn = get_connection()
        conn.run(
            f"DELETE FROM {self.table} WHERE session_id = :session_id",
            session_id=session_id,
        )
        conn.close()


memory = SessionMemory()
print("SessionMemory ready.")

SessionMemory ready.


In [0]:
import re


class DemoAgent:
    """A tiny deterministic agent — reliable in a live training session.

    On every turn it:
      1. Retrieves the full prior history from Lakebase (its "context"),
      2. Computes a reply with simple rules — in a REAL agent this is
         exactly where you would call an LLM, passing the retrieved
         history as conversation context,
      3. Persists BOTH the user's message and its own reply to Lakebase.
    """

    def __init__(self, session_id, memory):
        self.session_id = session_id
        self.memory = memory

    def reply(self, user_text):
        # 1) Retrieve prior turns from Postgres — this is the agent's memory.
        history = self.memory.get_history(self.session_id)
        # Everything ever said in this session (used for recall lookups).
        all_text = " ".join(t["content"] for t in history) + " " + user_text

        text = user_text.lower()
        name_stmt = re.search(r"(?:my name is|i'?m)\s+(\w+)", text)
        like_stmt = re.search(r"\bi like (.+?)(?:[.?!]|$)", text)
        asks_name = ("my name" in text or "who am i" in text) and not name_stmt
        asks_likes = "what do i like" in text

        # 2) Deterministic reply logic.
        #    (In a real agent, replace this block with an LLM call that
        #     receives the retrieved history + the new message as context.)
        if name_stmt:
            name = name_stmt.group(1).capitalize()
            response = f"Nice to meet you, {name}! I'll remember that."
        elif like_stmt:
            likes = like_stmt.group(1).strip()
            response = f"Got it — noted that you like {likes}."
        elif asks_name and asks_likes:
            m_name = re.search(r"(?:my name is|i'?m)\s+(\w+)", all_text, re.IGNORECASE)
            m_like = re.search(r"\bi like (.+?)(?:[.?!]|$)", all_text, re.IGNORECASE)
            response = (
                f"You're {m_name.group(1).capitalize()}, and you like "
                f"{m_like.group(1).strip()} — I remembered from our conversation!"
            )
        elif asks_name:
            m_name = re.search(r"(?:my name is|i'?m)\s+(\w+)", all_text, re.IGNORECASE)
            response = (
                f"You're {m_name.group(1).capitalize()} — I remembered from our conversation!"
                if m_name
                else "You haven't told me your name yet."
            )
        elif asks_likes:
            m_like = re.search(r"\bi like (.+?)(?:[.?!]|$)", all_text, re.IGNORECASE)
            response = (
                f"You told me you like {m_like.group(1).strip()}."
                if m_like
                else "You haven't told me what you like yet."
            )
        else:
            response = "Interesting! Tell me more."

        # 3) Persist both sides of the conversation to Postgres.
        next_turn = (history[-1]["turn"] + 1) if history else 1
        self.memory.save_turn(self.session_id, next_turn, "user", user_text)
        self.memory.save_turn(self.session_id, next_turn + 1, "agent", response)
        return response


print("DemoAgent ready.")

DemoAgent ready.


In [0]:
# --- Session 1: a first conversation ---
# The user introduces themselves and states a preference.
# Every turn (user AND agent) is saved to Lakebase.
SESSION_ID = "demo-session-1"

agent = DemoAgent(SESSION_ID, memory)

for user_message in ["Hi, I'm Alan", "I like tea."]:
    print(f"User : {user_message}")
    print(f"Agent: {agent.reply(user_message)}\n")

print("What the agent just stored in Postgres:")
for t in memory.get_history(SESSION_ID):
    print(f"  turn {t['turn']:>2} [{t['role']:>5}] {t['content']}")

User : Hi, I'm Alan
Agent: Nice to meet you, Alan! I'll remember that.

User : I like tea.
Agent: Got it — noted that you like tea.

What the agent just stored in Postgres:
  turn  1 [ user] Hi, I'm Alan
  turn  2 [agent] Nice to meet you, Alan! I'll remember that.
  turn  3 [ user] I like tea.
  turn  4 [agent] Got it — noted that you like tea.


In [0]:
# --- Simulate a total agent restart ---
# Destroy the in-process agent object. A brand-new instance has ZERO
# in-memory state — exactly what a serverless cold start, a crash, or a
# redeploy would look like.
del agent

agent2 = DemoAgent(SESSION_ID, memory)  # same session, fresh "process"

user_message = "What's my name? What do I like?"
print(f"User : {user_message}")
print(f"Agent: {agent2.reply(user_message)}")
print()
print("The new agent instance answered correctly — memory lives in Lakebase, not in the process.")

# Show the raw table: everything the agent has ever stored.
conn = get_connection()
rows = conn.run(
    "SELECT session_id, turn, role, content, created_at "
    "FROM agent_session_memory ORDER BY session_id, turn"
)
conn.close()
print("\nRaw table contents (agent_session_memory):")
for row in rows:
    print(f"  {row[0]} | turn {row[1]:>2} | {row[2]:>5} | {row[3]} | {row[4]}")

User : What's my name? What do I like?
Agent: You're Alan, and you like tea — I remembered from our conversation!

The new agent instance answered correctly — memory lives in Lakebase, not in the process.

Raw table contents (agent_session_memory):
  demo-session-1 | turn  1 |  user | Hi, I'm Alan | 2026-10-26 18:32:31.076544+00:00
  demo-session-1 | turn  2 | agent | Nice to meet you, Alan! I'll remember that. | 2026-10-26 18:32:31.330122+00:00
  demo-session-1 | turn  3 |  user | I like tea. | 2026-10-26 18:32:31.839571+00:00
  demo-session-1 | turn  4 | agent | Got it — noted that you like tea. | 2026-10-26 18:32:32.089392+00:00
  demo-session-1 | turn  5 |  user | What's my name? What do I like? | 2026-10-26 18:32:33.060643+00:00
  demo-session-1 | turn  6 | agent | You're Alan, and you like tea — I remembered from our conversation! | 2026-10-26 18:32:33.302594+00:00


## Takeaways

* **Memory survives restarts.** A brand-new agent instance — a cold start, a crash, a redeploy — picks up right where the last one left off, because every turn lives in Postgres.
* **Agents scale horizontally.** Any replica can serve any request: they all read and write the same Lakebase database, so there's one consistent conversation state.
* **Idle agents cost ~nothing.** Lakebase scales to zero, so a low-traffic agent's memory store bills pennies until a connection wakes it (~100 ms).
* **No passwords in code.** Access is OAuth, minted from your Databricks identity. Tokens expire after 1 hour, which we sidestep by opening a fresh connection per operation.

### Where to take this next

* **pgvector for semantic recall** — embed turns and retrieve only the most relevant memories as context, instead of replaying the whole history.
* **Session TTLs and cleanup jobs** — expire old sessions automatically instead of letting the table grow forever.
* **Branching for schema changes** — copy-on-write branches let you test a new memory-table design on a snapshot of real data.
* **Read-only replicas** — offload heavy retrieval workloads (e.g., large context retrieval) to a second endpoint.